# LLM Prompting

There are lots of things that gets hidden from us when using a LLM chatbot, not talking just about the math that goes on but even simpler stuff like it being stateless, system prompts, temperature... I know a bit about those but not enough, so let's dive in.

## Statelessness
So when we call a model, it process the individual message we sent. The previous messages are not stored. Let me try it out.

In [1]:
from constants import MODEL
from litellm import completion

response = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Just wanted to let you know that my name is Rafael. Can you tell me in one line, what might be the origin of my name?"}]
)

In [5]:
response.choices[0].message.content

'Rafael is a name of Hebrew origin, derived from "Rapha\'el," meaning "God has healed," and is widely used in Spanish, Portuguese, and other Romance-speaking cultures.'

In [6]:
response_new = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Summarize the origin of my name in just one word."}]
)

response_new.choices[0].message.content

'Please provide your name so I can give an accurate one-word summary of its origin.'

Ok so as we can see it doesnt remember my name. Let me try to chain the messages now.

In [8]:
chained_response = completion(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Just wanted to let you know that my name is Rafael. Can you tell me in one line, what might be the origin of my name?"
        },
        response.choices[0].message,
        {
            "role": "user",
            "content": "Summarize the origin of my name in just one word."
        }
    ]
)

chained_response.choices[0].message.content

'Hebrew.'

Cool. I guess litellm also provides a class for messages, so no need to store as dict... maybe?

In [9]:
from litellm import Message # oh yeah

So with that, I can create some simple functions that first store the messages, then send the whole batch.

In [14]:
messages: list[Message] = []

def send_user_message(message: str, model: str = MODEL):
    messages.append(Message(content=message, role="user"))
    response = completion(model=model, messages=messages)
    messages.append(response.choices[0].message)
    return response

In [15]:
response = send_user_message("In just a paragraph, what the origin of the name Rafael?")
response.choices[0].message.content

'The name **Rafael** originates from the Hebrew name **Rapha\'el**, derived from the roots *rā\'ā* ("to heal") and *Elohīm* ("God"), meaning "God heals" or "God has healed." It appears in the Bible, notably in the Book of Ezekiel as one of the four cherubim, and is associated with the archangel Raphael, a key figure in Jewish, Christian, and Islamic traditions who is believed to guide souls and heal. The name has been widely adopted across cultures, particularly in Spanish-speaking countries, and retains its biblical and spiritual significance.'

In [16]:
response = send_user_message("Can you summarize to just a very short phrase?")
response.choices[0].message.content

'Hebrew name meaning "God heals," associated with the archangel Raphael.'

In [17]:
response = send_user_message("No, make it short. Make it a SINGLE word.")
response.choices[0].message.content

'Hebrew.'

Ok that pretty much covers this.

## System Prompts

So this is interesting. This is a way to create a system side definition for the model to follow, like explaining a specific style of answer you want it to follow. I've heard that system prompt cannot be manipulated by the user, but I'll try either way.